# NLP

## Imports de librerías

In [1]:
%%capture
!pip install -U spacy
!python -m spacy download en_core_web_md

In [2]:
from functools import partial

In [3]:
import os
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd 
import numpy as np

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
# <- Machine learning ->
from sklearn.metrics import classification_report

In [7]:
# <- Analisis y tratamiento de datos tipo texto ->
import spacy

## Funciones personalizadas

In [8]:
def clean_normalize_text(doc: str = None, tokenizer=True, c_stop_w=False, c_punct=False, lemmatizer=False, lower=True):
    doc_clean_normalize = []
    if tokenizer and doc:
        for token in nlp(doc):
            # Primero tomamos el texto normal
            token_clean_norm = token.text

            # Saltar stopwords
            if c_stop_w and token.is_stop:
                continue

            # Saltar signos de puntuación
            if c_punct and token.is_punct:
                continue

            # Aplicar lematización si se solicita
            if lemmatizer:
                token_clean_norm = token.lemma_

            # Aplicar minúsculas si se solicita
            if lower:
                token_clean_norm = token_clean_norm.lower()

            doc_clean_normalize.append(token_clean_norm)

    return ' '.join(doc_clean_normalize)

## Cargar datasets

In [9]:
%%capture
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [10]:
%%capture
!tar xvzf ./aclImdb_v1.tar.gz

In [11]:
dir_base = './aclImdb'
neg_reviews = []

for folder in ['test', 'train']:
  dir = os.path.join(dir_base, folder, 'neg')
  for f in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, f)):
      with open(file=os.path.join(dir, f),encoding='UTF-8', mode='r') as review:
        neg_reviews.append(review.read())

In [12]:
dir_base = './aclImdb'
pos_reviews = []

for folder in ['test', 'train']:
  dir = os.path.join(dir_base, folder, 'pos')
  for f in os.listdir(dir):
    if os.path.isfile(os.path.join(dir, f)):
      with open(file = os.path.join(dir, f), mode='r', encoding='UTF-8') as review:
          pos_reviews.append(review.read())

In [13]:
df_pos_reviews = pd.DataFrame({
    'reviews':pos_reviews,
    'tag':'pos'
})

df_neg_reviews = pd.DataFrame({
    'reviews':neg_reviews,
    'tag':'neg'
})

df_reviews = pd.concat([df_neg_reviews, df_pos_reviews], join='outer', axis=0)

## Preprocesamiento para NLP

In [14]:
nlp = spacy.load("en_core_web_md")

In [15]:
pos_samples = df_pos_reviews.iloc[:1000, :]
neg_samples = df_neg_reviews.iloc[:1000, :]

df_sample = pd.concat([pos_samples, neg_samples], axis=0, join='outer')

In [16]:
# creamos una funcion parcial para definir los valores por defecto en los parametros de la función 'clean_normalize_text'

clean_normalize_text_partial = partial(clean_normalize_text, tokenizer=True, c_stop_w=True, c_punct=True, lemmatizer=True, lower=True)

# aplicamos la función parcial con los parametros por defecto que le definimos
df_sample["reviews"] = df_sample["reviews"].apply(clean_normalize_text_partial)

### Separación en variables predictoras y target

In [17]:
X = df_sample['reviews']
y = df_sample['tag']

### Separación de subconjuntos de entrenamiento y testeo
Generar X_train, X_test, y_train e y_test de nuevo, ya que las modificamos anteriormente con el count vectorizer:

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3)

## Vectorización TF-IDF

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html


Ahora, importar tfidf vectorizer y aplicarlo sobre X_train y X_test

In [19]:
tfidf_vec = TfidfVectorizer()
tfidf_vec.fit(X_train)

TfidfVectorizer()

In [20]:
X_train = tfidf_vec.transform(X_train)

In [21]:
X_test = tfidf_vec.transform(X_test)

## MLP

Dado que el resultado de la vectorización clasica, nos da un dataframe de alta dimensionalidad y relaciones muy complejas y ocultas entre variables, es una excelente oportunidad para comprobar como se comporta un perceptron multicapa.

In [22]:
%%capture
!pip3 install keras tensorflow

In [23]:
from keras import Input, Sequential
from keras.layers import Dense
from keras.optimizers import Adam

In [24]:
from sklearn.preprocessing import LabelBinarizer

In [25]:
import matplotlib.pyplot as plt

### Preprocesamiento adicional para nuestro MLP

Como vimos, algo que diferencia a las redes neuronales de algortimos clasicos. de ML es la no necesidad de feature engineer, pero entiendo esta como la creación de columnas derivadas en varse aciertos patrones; el preprocesamiento y limpieza seguiran siendo super necesarios para que la red pueda aprender.

Nuestro MLP necesitara la columna target en formato numerico, por lo que previamente la binarizaremos.

In [26]:
lb = LabelBinarizer()
lb.fit(y_train)

y_train = lb.transform(y_train)
y_test = lb.transform(y_test)

X_train debe estar en formato denso (no sparse), lo convertiremos a tal formato:

In [27]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 118455 stored elements and shape (1400, 17732)>

In [28]:
X_train = X_train.toarray()
X_train = X_train.astype('float32')

### Definiendo las diferentes capas del MLP:

- Capa de entrada: Tendra x caracteristicas de entrada igual a la cantidad. de features / caracteristicas del dataset vectorizado (cantidad de palabras en el vocavulario)
- Capas ocultas: Dado la complegidad y dimension del conjunton de datos, estara compuesto por tres capas ocultas de x neuronas cada una.
- Capa de salida: Como es un problema. declasificación binario, se utilizara una sola neurona que predicira la probabilidad de la clase positiva.

Capas de entrada

In [29]:
capa_entrada = Input(shape=(X_train.shape[1],))

Capas ocultas

In [30]:
cantidad_neuronas_capa = int((X_train.shape[1] + len(np.unique(y_train)) / 2))

In [31]:
capa_oculta_uno = Dense(units=cantidad_neuronas_capa, activation='relu')

In [32]:
capa_oculta_dos = Dense(units=cantidad_neuronas_capa, activation='relu')

In [33]:
capa_oculta_tres = Dense(units=cantidad_neuronas_capa, activation='relu')

In [34]:
capa_oculta_cuatro = Dense(units=cantidad_neuronas_capa, activation='relu')

In [35]:
capa_oculta_cinco = Dense(units=cantidad_neuronas_capa, activation='relu')

Capas de salida

In [36]:
capa_salida = Dense(units=1, activation='sigmoid')

### Orquestación de las neuronas dentro del modelo de red sequencial:

Dado que el perceptron es un modelo de red neuronal donde todas las neuronas estan **secuencialmente** y **densamente** o totalmente conectadas con la capa siguiente, etonces utilizaremos el modelo `Sequential` para conectar las capas densas una despues de la otra.

In [37]:
mlp_model = Sequential(layers= [
    capa_entrada,
    capa_oculta_uno,
    capa_oculta_dos,
    capa_oculta_tres,
    capa_oculta_cuatro,
    capa_oculta_cinco,
    capa_salida
]
)

In [38]:
mlp_model.compile(optimizer=Adam(0.01), loss='BinaryCrossentropy')

Entrenamiento

In [ ]:
historial = mlp_model.fit(X_train, y_train, epochs=5, validation_split=0.2, batch_size=700)

Epoch 1/5


In [ ]:
y_pred = mlp_model.predict(X_test)

In [ ]:
y_pred = (y_pred >= 0.5).astype(int)

In [ ]:
print(classification_report(y_pred=y_pred, y_true=y_test))

In [ ]:
historial.history.keys()

In [ ]:
error_train = historial.history['loss']
error_val = historial.history['val_loss']

In [ ]:
plt.figure(figsize=(10, 10))
plt.plot(error_train, label='train loss', color='r')
plt.plot(error_val, label='val loss', color='b')
plt.legend()
plt.show()

<font color="gray" size="2">

---

© Yané, Ian Cristian Ariel — CC BY-NC 4.0  
Material de autoría original. Uso no comercial. Requiere atribución.  
Ver licencia completa en el repositorio.

</font>